# Module 2.5 - Building a RAG Chatbot

**Reference:** [`05-building-a-rag-chatbot.md`](./05-building-a-rag-chatbot.md)

## What you'll do in this notebook

- Wire ChromaDB, OpenAI embeddings, and chat completions into a single `RAGChatbot` class.
- Write a prompt that forces the model to answer **only** from retrieved context.
- Add source citations to every reply.
- Test a "don't know" edge case where retrieval returns nothing useful.
- Move your capstone project forward: add RAG over your own documents.

## Setup

In [ ]:
import os
import shutil
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBED_MODEL = "text-embedding-3-small"
CHROMA_PATH = "./chroma_store_m2_5"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

chroma = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"OK: client ready, chat model = {MODEL}")

## Exercise 1 - The RAGChatbot class

Three methods: `add_documents(texts, metadatas)`, `retrieve(query, top_k)`, and `chat(query, top_k)`. The last two are where the real work lives.

In [ ]:
class RAGChatbot:
    def __init__(self, collection_name: str = "rag_docs"):
        self.collection = chroma.get_or_create_collection(collection_name)

    # -- indexing --------------------------------------------------------
    def add_documents(self, texts: list[str], metadatas: list[dict]) -> None:
        # TODO:
        # 1. Embed all texts in one batch call.
        # 2. Build ids like "doc_<existing_count>_<i>".
        # 3. self.collection.add(...) with ids, embeddings, documents, metadatas.
        raise NotImplementedError

    # -- retrieval -------------------------------------------------------
    def retrieve(self, query: str, top_k: int = 3) -> tuple[list[str], list[dict]]:
        # TODO:
        # 1. Embed the query.
        # 2. Call self.collection.query(query_embeddings=[...], n_results=top_k).
        # 3. Return (documents, metadatas) for the first (and only) query row.
        raise NotImplementedError

    # -- generation ------------------------------------------------------
    def chat(self, query: str, top_k: int = 3) -> dict:
        documents, metadatas = self.retrieve(query, top_k=top_k)

        context_block = "\n\n".join(
            f"[{i+1}] source={m.get('source', 'unknown')}\n{doc}"
            for i, (doc, m) in enumerate(zip(documents, metadatas))
        )

        system = (
            "You answer questions using ONLY the provided context. "
            "If the context does not contain enough information, reply exactly: "
            "\"I don't have enough information to answer that.\" "
            "When you do answer, cite the source in square brackets like [1] or [2]."
        )
        user = f"Context:\n{context_block}\n\nQuestion: {query}"

        # TODO: call client.chat.completions.create with model=MODEL,
        #   messages=[{system}, {user}], temperature=0.2.
        # Return {"answer": <reply text>, "sources": [m.get('source') for m in metadatas]}.
        raise NotImplementedError

print("RAGChatbot skeleton ready")

## Exercise 2 - Index and query a small corpus

We'll use a handful of short "documents" so you can see retrieval behaviour clearly. Swap these for your own domain later.

In [ ]:
DOCS = [
    "Python was created by Guido van Rossum and first released in 1991.",
    "Python 3.0 was released in 2008 and was not backwards-compatible with Python 2.",
    "JavaScript was created by Brendan Eich at Netscape in 1995.",
    "Rust was designed at Mozilla Research; its first stable release was 1.0 in 2015.",
    "DevBrew Cafe is open 06:00-22:00 Monday to Friday and 08:00-20:00 at weekends.",
    "DevBrew accepts Visa, Mastercard, American Express, and Apple Pay; cash up to 50 pounds.",
]
METAS = [
    {"source": "python_history.md"},
    {"source": "python_history.md"},
    {"source": "javascript_history.md"},
    {"source": "rust_history.md"},
    {"source": "devbrew_handbook.md"},
    {"source": "devbrew_handbook.md"},
]

bot = RAGChatbot(collection_name="rag_demo")
bot.add_documents(DOCS, METAS)

for question in [
    "When was Python first released?",
    "Can I pay by Apple Pay at DevBrew?",
    "What's the airspeed velocity of an unladen swallow?",
]:
    print(f"\nQ: {question}")
    result = bot.chat(question)
    print(f"A: {result['answer']}")
    print(f"   sources: {result['sources']}")

**What to look for:**

- The Python and DevBrew questions should produce grounded answers with citations.
- The swallow question should produce the *exact* fallback: "I don't have enough information to answer that." If it doesn't, tighten your prompt.

## Exercise 3 - Retrieval failure modes

Swap in the **wrong** embedding model for the query (while keeping documents indexed with `text-embedding-3-small`) and try a query that worked before. Spoiler: it won't work any more.

This drives home the rule from the reading: queries and documents must be embedded with the same model.

In [ ]:
# TODO:
# Temporarily monkey-patch RAGChatbot.retrieve to embed the query with
# model="text-embedding-3-large" instead of EMBED_MODEL, then run the Python
# question again. Record how the top result changes.
#
# Remember to undo the monkey-patch after the experiment.


## Capstone checkpoint 2

Your capstone chatbot now needs to talk to documents, not just generic knowledge.

1. Collect at least 10 short documents for your chosen domain.
2. Run each through the `process_document` pipeline from section 3 (chunking + embeddings).
3. Load them into a fresh ChromaDB collection via `RAGChatbot.add_documents`.
4. Update your capstone `chat` method to use retrieval before calling the LLM.
5. Write 5-10 test queries whose expected source document you know. Count how many are correctly cited.

You'll revisit retrieval quality systematically when you build the evaluation harness in Module 3.5.

## What's next

Retrieval is rarely perfect on the first try. Module 3 introduces techniques for improving it: agentic multi-step retrieval, reranking, hybrid (vector + keyword) search, semantic caching to cut costs, and a proper evaluation harness. Start with `module_3/01-agentic-rag.ipynb`.